# 06 — Validation Strategy Upgrade (Time-Aware Modeling)

## 🎯 Objective
Upgrade the validation strategy from GroupKFold to a **time-aware validation scheme** that simulates real-world deployment.

## Why This Matters
- Loan data is **time-dependent**
- Models must predict **future defaults using past data**
- Prevents overly optimistic performance estimates

## Key Upgrade
From:
- GroupKFold (no customer leakage)

To:
- Time-based validation (no future leakage)

## Expected Outcome
- Lower but **more realistic F1 score**
- Improved generalization (Kenya → Ghana)

**1.IMPORTS**

In [1]:
# ======================
# IMPORTS
# ======================
import sys, os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np

from sklearn.metrics import f1_score

import lightgbm as lgb

from src.pipeline import full_pipeline
from src.data_loader import load_data
from src.config import load_config, set_seed

import warnings
warnings.filterwarnings("ignore")

**2. LOAD CONFIG**

In [2]:
config = load_config()

N_SPLITS = config["training"]["n_splits"]
RANDOM_STATE = config["training"]["random_state"]

set_seed(RANDOM_STATE)

print(config)

{'model': {'type': 'lightgbm'}, 'training': {'n_splits': 5, 'random_state': 42}, 'features': {'use_economic_data': True, 'use_customer_aggregates': True, 'use_loan_aggregates': True}, 'threshold': {'optimize': True}}


**3. LOAD DATA + PIPELINE**

In [3]:
train, test, _ = load_data()

train_processed, test_processed = full_pipeline(train, test)

print("Train shape:", train_processed.shape)

Train shape: (68654, 26)


**4. FEATURE SELECTION (LEAKAGE-SAFE)**

In [4]:
DROP_COLS = [
    "ID",
    "target",
    "customer_id",
    "tbl_loan_id",
    "lender_id",
    "disbursement_date",
    "due_date"
]

FEATURES = [col for col in train_processed.columns if col not in DROP_COLS]

X = train_processed[FEATURES]
y = train_processed["target"]

print("Number of features:", len(FEATURES))

Number of features: 19


**5. SORT BY TIME (KEY STEP)**

In [5]:
train_processed = train_processed.sort_values("disbursement_date").reset_index(drop=True)

X = train_processed[FEATURES]
y = train_processed["target"]

**6. TIME-BASED SPLIT FUNCTION**

In [6]:
def time_based_split(df, n_splits=5):
    """
    Expanding window time-based split:
    Train on past → validate on future
    """
    n_samples = len(df)
    fold_size = n_samples // n_splits

    for i in range(1, n_splits):
        train_end = fold_size * i
        val_end = fold_size * (i + 1)

        train_idx = df.index[:train_end]
        val_idx = df.index[train_end:val_end]

        yield train_idx, val_idx

**7. CLASS IMBALANCE**

In [7]:
pos = y.sum()
neg = len(y) - pos

scale_pos_weight = neg / pos

print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 53.573926868044516


**8. LIGHTGBM MODEL**

In [8]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

**9. TIME-BASED CV FUNCTION**

In [9]:
from lightgbm import early_stopping, log_evaluation

def run_time_cv(model, X, y, df):
    scores = []

    for fold, (train_idx, val_idx) in enumerate(time_based_split(df, N_SPLITS)):
        
        X_train, X_val = X.loc[train_idx], X.loc[val_idx]
        y_train, y_val = y.loc[train_idx], y.loc[val_idx]

        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[early_stopping(50), log_evaluation(0)]
        )

        preds = model.predict(X_val)
        score = f1_score(y_val, preds)

        scores.append(score)

        print(f"Fold {fold}: F1 = {score:.5f}")

    print("\nMean F1:", np.mean(scores))
    return np.mean(scores)

**10. RUN MODEL**

In [10]:
print("🚀 Time-Based Validation Results:")
time_score = run_time_cv(lgb_model, X, y, train_processed)

🚀 Time-Based Validation Results:
[LightGBM] [Info] Number of positive: 255, number of negative: 13475
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006267 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1878
[LightGBM] [Info] Number of data points in the train set: 13730, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.018572 -> initscore=-3.967328
[LightGBM] [Info] Start training from score -3.967328
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[75]	valid_0's binary_logloss: 0.0137835
Fold 0: F1 = 0.83486
[LightGBM] [Info] Number of positive: 442, number of negative: 27018
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007536 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1987
[LightGBM] [Info] Number of data points in the train

**11.COMPARE WITH GROUPKFOLD**

In [11]:
from sklearn.model_selection import GroupKFold

def run_group_cv(model, X, y, groups):
    scores = []
    gkf = GroupKFold(n_splits=N_SPLITS)

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(X_train, y_train)
        preds = model.predict(X_val)

        score = f1_score(y_val, preds)
        scores.append(score)

    return np.mean(scores)


group_score = run_group_cv(
    lgb_model,
    X,
    y,
    train_processed["customer_id"]
)

print("GroupKFold F1:", group_score)
print("Time-based F1:", time_score)

[LightGBM] [Info] Number of positive: 972, number of negative: 53951
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005889 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2204
[LightGBM] [Info] Number of data points in the train set: 54923, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.017698 -> initscore=-4.016476
[LightGBM] [Info] Start training from score -4.016476
[LightGBM] [Info] Number of positive: 1005, number of negative: 53918
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004432 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2218
[LightGBM] [Info] Number of data points in the train set: 54923, number of used features: 18
[LightGBM] [Info] [bi

**13. COMPARE RESULTS**

In [12]:
results = pd.DataFrame({
    "Validation Strategy": ["GroupKFold", "Time-Based"],
    "F1 Score": [group_score, time_score]
})

display(results)

,Validation Strategy,F1 Score
0,GroupKFold,0.832438
1,Time-Based,0.792745


# 📊 Validation Strategy Upgrade — Summary Report

## 🎯 Objective
To upgrade the validation approach from GroupKFold to a time-aware validation strategy that reflects real-world deployment conditions and eliminates temporal leakage.

---

## 🔍 Key Results

| Validation Strategy | F1 Score |
|--------------------|--------|
| GroupKFold         | 0.8324 |
| Time-Based         | 0.7927 |

---

## 🧠 Key Insights

### 1. Realistic Performance Estimation

The transition to time-based validation resulted in a moderate performance drop (~4%), confirming that:

- GroupKFold slightly overestimates model performance
- Time-based validation provides a more realistic estimate of future predictive capability

---

### 2. Strong Generalization Ability

The relatively small gap between validation strategies indicates:

- Minimal data leakage
- Robust feature engineering pipeline
- Stable model behavior across time

This suggests that the model is learning genuine patterns rather than exploiting dataset artifacts.

---

### 3. Temporal Variability in Model Performance

Significant variation across folds (e.g., one fold dropping to ~0.56 F1) highlights:

- Potential temporal distribution shifts
- Changes in borrower behavior over time
- Sensitivity to economic or operational conditions

This reinforces the importance of time-aware validation in financial modeling.

---

### 4. Effective Handling of Class Imbalance

With a severe class imbalance (scale_pos_weight ≈ 53.6), the model maintains strong performance:

- F1 score remains high despite imbalance
- Indicates effective use of weighting strategy
- Confirms robustness of modeling approach

---

### 5. Clean and Leakage-Safe Feature Pipeline

Post-leakage audit:

- High-risk features removed (e.g., default_rate, customer_default_rate)
- Remaining features are causally valid
- Model relies on behavioral and financial signals

---

## 🚨 Critical Insight

> Models evaluated without temporal constraints may appear strong but fail in production.

This upgrade ensures alignment between validation and real-world deployment.

---

## 🧠 Final Conclusion

The time-based validation framework provides:

- Reliable performance estimation
- Improved robustness
- Production-aligned evaluation

The model demonstrates strong predictive capability with minimal reliance on leakage-prone features.

---

## 🚀 Next Steps

1. Hyperparameter tuning (Optuna)
2. Threshold optimization for F1 maximization
3. Incorporation of economic indicators
4. Final model training and submission

---



---

> “Validation strategy is as important as the model itself.”